In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import date, datetime
import plotly.express as px
import plotly.graph_objects as go

In [4]:
call_folder = '../option_data/calls/combined/'
call_data = pd.read_csv(call_folder + 'combined_calls.csv')

put_folder = '../option_data/puts/combined/'
put_data = pd.read_csv(put_folder + 'combined_puts.csv')


In [37]:
call_data['spread'] = call_data['ask'] - call_data['bid']

date_of_collection = pd.to_datetime('2025-07-08')
call_data['expiry'] = pd.to_datetime(call_data['expiry'], errors='coerce')
call_data['time_to_expiry'] = call_data['expiry']-date_of_collection

call_data['moneyness'] = (210.01 - call_data['strike']) / 210.01


<a id="Introduction"></a>
# Introduction

The Dataset contains 381 observations of call options collected on <u>July 8th 2024</u>. <br>
The Underlying Stock Price was <u>$210.01</u>

It includes the following variables:
- `contract_name`: A unique identifier for the specific option contract. It includes the underlying stock symbol, the expiration date, option type, and the strike price 
  - For Example: AAPL-250801-C-00110000:
    - AAPL: Underlying Stock Symbol
    - 250801: Expiration Date (01 Aug 2025)
    - C: Option Type (Call)
    - 00110000: Strike Price (Expressed in Thousandth of a Dollar), Strike = 110

- `last_trade_date_(gmt-4)`: The date at which the option contract was most recently traded in GMT-4

- `strike`: Strike Price

- `last_price`: The price at which the option contract was most recently traded. 

- `bid`: Bid Price

- `ask`: Ask Price

- `change`: The difference between the current Last Price and the previous day's closing price for that option contract.

- `pct_change`: The percentage change in the option's price from the previous day's closing price.

- `volume`: The total number of option contracts that have been traded for that specific option during the current trading day.

- `open_interest`: The total number of outstanding option contracts for a particular strike price and expiration date that have not yet been closed out, exercised, or expired. 

- `implied_volatility`: Implied Volatility

- `expiry`: Date on which the options contract expires.
- `spread`: Difference in Bid and Ask Price
- `time_to_expiry`: Number of days from July 8th 2024 to Expiry
- `moneyness`: Relationship between the option's Strike Price and the Underlying Stock Price at the time of collection

<a id="Data Selection"></a>

# Data Selection <br>
To ensure the integrity of this analysis, I applied the following selection criteria to the raw options data: <br>

- Implied Volatility  <br>
  - IV > 0
  - Many of the Options have an IV of 0, this could be because there were no recent trades at the point of collection. As such, we will be omitting rows where IV = 0

- Near-The-Money Trades <br>
  - Moneyness Delta between 0.30 and 0.70
  - These contracts are the most frequently traded, leading to the tightest bid-ask spreads and the most accurate market pricing

- Bid Prices <br>
  - Bid > 0
  - These contracts are actively being traded, ensuring 2-sided market value

In [38]:
iv_filter = call_data['implied_volatility'] > 0
moneyness_filter = call_data['moneyness'] > 0.1

bid_filter = call_data['bid'] > 0

filtered_call_data = call_data[bid_filter & iv_filter & moneyness_filter]

In the end, we are only left with 7 rows that fulfill our data selection criterias <br>

This is an notable area of improvement for future iterations, but for now, we will carry on our analysis using our small sample

In [57]:
filtered_call_data

,contract_name,last_trade_date_(gmt-4),strike,last_price,bid,ask,change,pct_change,volume,open_interest,implied_volatility,expiry,spread,time_to_expiry,moneyness
139,AAPL250919C00015000,05/06/2025 1:04 pm,15.0,186.75,198.70,199.30,0.0,0.0,1.0,0.0,4.6602,2025-09-19,0.60,73 days,0.928575
146,AAPL250919C00050000,30/05/2025 9:36 am,50.0,149.50,160.85,161.50,0.0,0.0,1.0,664.0,1.8066,2025-09-19,0.65,73 days,0.761916
263,AAPL251121C00110000,03/06/2025 1:48 pm,110.0,95.85,103.90,105.90,0.0,0.0,1.0,0.0,0.9054,2025-11-21,2.00,73 days,0.476215
316,AAPL251219C00025000,25/02/2025 10:37 am,25.0,221.35,191.30,195.20,0.0,0.0,1.0,1.0,3.0542,2025-12-19,3.90,73 days,0.880958
327,AAPL251219C00080000,06/06/2025 12:09 pm,80.0,124.23,133.60,135.35,0.0,0.0,4.0,0.0,1.1283,2025-12-19,1.75,73 days,0.619066
329,AAPL251219C00090000,03/06/2025 9:47 am,90.0,114.55,124.10,125.95,0.0,0.0,1.0,0.0,1.0410,2025-12-19,1.85,73 days,0.571449
334,AAPL251219C00115000,03/06/2025 11:23 am,115.0,90.94,99.60,102.25,0.0,0.0,1.0,232.0,0.8264,2025-12-19,2.65,73 days,0.452407
